# Full Ink-System Probabilistic Diagnosis Demo (v6.3)

This notebook uses the **full KB** and includes a **pre-visit parts list** feature.

## Pre-visit parts list policy (your choices)
- Show **top 5** parts to prepare
- **Parts only** (no tools/consumables)
- Show **alternatives** only when the corresponding root cause is **highly probable**

Important note:
- Your provided parts CSV does **not** include ink-tube indices `Z-0502...` mentioned in diagnostics.
  Those remain as placeholders until a tube-specific parts list is provided.


In [1]:
import json, math, uuid, datetime
from pathlib import Path

KB_PATH = Path(r"C:/Project/CPP_CORE/kb/kb_ink_system_full_v6_3.json")
kb = json.loads(KB_PATH.read_text(encoding="utf-8"))

root_causes = {rc["root_cause_id"]: rc for rc in kb["catalogs"]["root_causes"]}
obs_catalog = {o["observation_id"]: o for o in kb["catalogs"]["observations"]}
procedures = {p["procedure_id"]: p for p in kb["catalogs"]["procedures"]}
error_models = {m["error_code"]: m for m in kb["catalogs"]["error_code_models"]}
parts = {p["part_id"]: p for p in kb["catalogs"].get("parts", [])}

likelihoods = kb["parameters"].get("likelihoods", {})
action_success = kb["parameters"].get("action_success", {})

cfg = kb.get("inference_config", {})
EPSILON = cfg.get("stop_condition", {}).get("epsilon_default", 0.02)

pv_cfg = cfg.get("previsit_parts", {})
PV_TOPK = pv_cfg.get("top_k", 5)
PV_ALT_TH = pv_cfg.get("high_probability_threshold_for_alternatives", 0.20)

print("Loaded KB:", kb["meta"]["name"])
print("Error codes:", len(error_models), "| Causes:", len(root_causes), "| Procedures:", len(procedures), "| Parts:", len(parts))
print("Previsit policy: top_k =", PV_TOPK, "| alt_threshold =", PV_ALT_TH)


Loaded KB: Ink system KB full (v6.3, previsit top-5 parts + alternatives policy)
Error codes: 41 | Causes: 54 | Procedures: 115 | Parts: 55
Previsit policy: top_k = 5 | alt_threshold = 0.2


In [2]:
def normalize(dist: dict) -> dict:
    s = sum(dist.values())
    if s <= 0:
        n = len(dist)
        return {k: 1.0/n for k in dist} if n else {}
    return {k: v/s for k, v in dist.items()}

def get_likelihood(obs_id: str, cause_id: str, value: str) -> float:
    vals = obs_catalog[obs_id]["values"]
    table = likelihoods.get(obs_id, {}).get(cause_id)
    if table is None:
        return 1.0 / len(vals)
    return float(table.get(value, 1.0 / len(vals)))

def update_belief(belief: dict, obs_id: str, value: str) -> dict:
    updated = {}
    for c, p in belief.items():
        updated[c] = p * get_likelihood(obs_id, c, value)
    return normalize(updated)

def initial_belief_from_error(error_code: str) -> dict:
    model = error_models[error_code]
    # Use the error-code prior but spread small mass over everything else
    alpha = cfg.get("upstream_generation", {}).get("alpha_error_code_mass", 0.90)
    ec_prior = normalize({cid: float(p) for cid, p in model.get("priors_root_cause_given_error", {}).items()})
    belief = {cid: 0.0 for cid in root_causes.keys()}
    other = [cid for cid in belief if cid not in ec_prior]
    other_mass = 1.0 - alpha
    if other:
        base = other_mass / len(other)
        for cid in other:
            belief[cid] += base
    for cid, p in ec_prior.items():
        belief[cid] += alpha * p
    return normalize(belief)

def apply_initial_evidence(belief: dict, evidence: dict) -> dict:
    for obs_id, val in evidence.items():
        if obs_id not in obs_catalog:
            continue
        belief = update_belief(belief, obs_id, val)
    return belief

def action_outcome_likelihood(action_id: str, cause_id: str, resolved_value: str) -> float:
    table = action_success.get(action_id, {}).get(cause_id)
    if table is None:
        return 0.45 if resolved_value == "yes" else 0.55
    return float(table.get(resolved_value, 0.5))

def expected_resolve_prob(belief: dict, repair_id: str) -> float:
    p = 0.0
    for c, bc in belief.items():
        p += bc * action_outcome_likelihood(repair_id, c, "yes")
    return p


## Pre-visit parts list

We estimate which parts may be needed **before diagnosis starts**, using:

- Initial belief over root causes (from error code + any initial evidence you have)
- Candidate **repair procedures** for that error code
- Each repair’s expected chance to resolve: `P(resolved=yes | belief, repair)`
- Mapping from repair → part(s) (from KB)

Then we rank parts and return **top 5**.

### Alternatives rule
If multiple repair options map to different parts, we show alternatives only when at least one linked root cause has probability ≥ 0.20.


In [3]:
def get_candidate_repairs(error_code: str):
    pids = error_models[error_code].get("candidate_procedure_ids", [])
    return [pid for pid in pids if procedures.get(pid, {}).get("kind") == "repair"]

def repair_parts(repair_id: str):
    proc = procedures.get(repair_id, {})
    # KB may store parts under different keys depending on version
    for key in ("part_ids", "parts", "requires_part_ids", "related_part_ids"):
        if key in proc and isinstance(proc[key], list):
            return proc[key]
    return []

def show_previsit(error_code: str, initial_evidence: dict = None):
    initial_evidence = initial_evidence or {}
    belief = apply_initial_evidence(initial_belief_from_error(error_code), initial_evidence)

    repairs = get_candidate_repairs(error_code)
    part_scores = {}
    part_sources = {}  # part_id -> list of repairs contributing

    for rid in repairs:
        p_res = expected_resolve_prob(belief, rid)
        parts_needed = repair_parts(rid)
        if not parts_needed:
            continue
        for pid in parts_needed:
            part_scores[pid] = part_scores.get(pid, 0.0) + p_res
            part_sources.setdefault(pid, []).append((rid, p_res))

    ranked = sorted(part_scores.items(), key=lambda kv: kv[1], reverse=True)
    top = ranked[:PV_TOPK]

    print("Pre-visit parts for error", error_code, "(top", PV_TOPK, ")")
    for pid, score in top:
        pinfo = parts.get(pid, {"part_id": pid, "description": "(placeholder / not in CSV)"})
        desc = pinfo.get("description") or pinfo.get("Description") or "(no description)"
        pn = pinfo.get("part_number") or pinfo.get("PartNumber") or ""
        print(f"- {pid} | {pn} | {desc}  (score={score:.3f})")
        # show alternatives only if high-probability causes exist
        # approximation: if any current top cause >= threshold, show source repairs
        if max(belief.values()) >= PV_ALT_TH:
            for rid, prs in sorted(part_sources.get(pid, []), key=lambda x: x[1], reverse=True)[:3]:
                print(f"    via {rid}: {procedures[rid]['label']} (p_resolve≈{prs:.3f})")

    return top


## Example: show pre-visit parts then run a session
Use `show_previsit(error_code, initial_evidence)` first.

For now, `initial_evidence` can contain warnings/counters if you have them.


In [4]:
# Example
error_code = "250001"
initial_evidence = {
    # Optional: if you know warnings/counters upfront
    "OBS_WARNING_290005_SEEN": "yes",
    "OBS_WARNING_290020_SEEN": "no",
    "OBS_LIFE_PUMP_TUBE": "high",
    "OBS_LIFE_INK_PUMP": "medium",
}

show_previsit(error_code, initial_evidence)


Pre-visit parts for error 250001 (top 5 )


[]